In [ ]:
from copy import deepcopy
import json
from pathlib import Path
from statistics import mean
import sys

import chess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Support opening the notebook from the repository root or scripts directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/chess_data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the DeepChess repository")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from chess_data import load_games, split_games, evaluated_positions, regression_metrics

def board_to_array(board):
    # Create an 19 x 8 x 8 array
    x = np.zeros((19, 8, 8), dtype=np.uint8)

    # Layers 0-11: Piece positions
    # White: pawn, knight, bishop, rook, queen, king
    # Black: pawn, knight, bishop, rook, queen, king
    piece_layer = {
        (chess.WHITE, chess.PAWN): 0,
        (chess.WHITE, chess.KNIGHT): 1,
        (chess.WHITE, chess.BISHOP): 2,
        (chess.WHITE, chess.ROOK): 3,
        (chess.WHITE, chess.QUEEN): 4,
        (chess.WHITE, chess.KING): 5,
        (chess.BLACK, chess.PAWN): 6,
        (chess.BLACK, chess.KNIGHT): 7,
        (chess.BLACK, chess.BISHOP): 8,
        (chess.BLACK, chess.ROOK): 9,
        (chess.BLACK, chess.QUEEN): 10,
        (chess.BLACK, chess.KING): 11,
    }

    for square, piece in board.piece_map().items():
        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)
        layer = piece_layer[(piece.color, piece.piece_type)]
        x[layer, row, col] = 1

    # Layers 12-15: Castling rights
    x[12, :, :] = int(board.has_kingside_castling_rights(chess.WHITE))
    x[13, :, :] = int(board.has_queenside_castling_rights(chess.WHITE))
    x[14, :, :] = int(board.has_kingside_castling_rights(chess.BLACK))
    x[15, :, :] = int(board.has_queenside_castling_rights(chess.BLACK))

    # Layer 16: Location of a pawn that can legally be
    # en passant
    if board.ep_square is not None:
        ep_square = board.ep_square
        row = 7 - chess.square_rank(ep_square)
        col = chess.square_file(ep_square)
        x[16, row, col] = 1

    # Layer 17: Side to move
    x[17, :, :] = int(board.turn == chess.WHITE)

    # Board shape
    x[18, :, :] = 1

    return x

In [ ]:
def make_dataset(games):
    arrays, labels = [], []
    for record in games:
        for board, score in evaluated_positions(record):
            arrays.append(board_to_array(board))
            labels.append(score)
    if not labels:
        raise ValueError("A split has no numeric evaluation labels")
    features = torch.from_numpy(np.stack(arrays)).float()
    targets = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)
    return TensorDataset(features, targets), labels

In [ ]:
INPUT_PATH = ROOT / "data/regressor_games.jsonl"
MAX_GAMES = None  # All games by default; set >= 3 for a small debugging run.
SEED = 42
BATCH_SIZE = 128
EPOCHS = 20
CHECKPOINT_PATH = None  # Optional path for saving each new validation-best model.

torch.manual_seed(SEED)
np.random.seed(SEED)
game_splits = split_games(load_games(INPUT_PATH, max_games=MAX_GAMES), seed=SEED)
prepared = {name: make_dataset(games) for name, games in game_splits.items()}
datasets = {name: pair[0] for name, pair in prepared.items()}
# Only training labels determine the constant prediction.
baseline_mean = mean(prepared["train"][1])
loaders = {
    name: DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=(name == "train"),
                     generator=torch.Generator().manual_seed(SEED))
    for name, dataset in datasets.items()
}
del prepared

In [ ]:
for name, dataset in datasets.items():
    print(f"{name}: {len(game_splits[name])} games, {len(dataset)} positions")
print(f"Training-mean baseline: {baseline_mean:.6f} pawns")

In [ ]:
class deepChess(nn.Module):
    def __init__(self):
        super(deepChess, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=19, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv2 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.fc1 = nn.Linear(in_features=128*8*8, out_features=1)

    def forward(self, a0):
        a1 = F.relu(self.conv1(a0))           
        a2 = F.relu(self.conv2(a1))   
        a3 = F.relu(self.conv3(a2))           
        a4 = F.relu(self.conv4(a3)) 
        flat = a4.view(a4.shape[0], 128*8*8)
        z5 = self.fc1(flat)  
        return z5

    
model = deepChess()
optimizer = optim.SGD(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

In [ ]:
def evaluate(model, loader):
    model.eval()
    actual, predicted = [], []
    with torch.no_grad():
        for features, targets in loader:
            actual.extend(targets.flatten().tolist())
            predicted.extend(model(features).flatten().tolist())
    return {
        "cnn": regression_metrics(actual, predicted),
        "baseline": regression_metrics(actual, [baseline_mean] * len(actual)),
    }


def train_model(model, optimizer, criterion, train_loader, validation_loader, epochs, checkpoint_path=None):
    if epochs < 1:
        raise ValueError("epochs must be positive")
    best_rmse = float("inf")
    best_state = None
    history = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for features, targets in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(features), targets)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite training loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0, error_if_nonfinite=True)
            optimizer.step()
            total_loss += loss.item() * len(targets)
        validation = evaluate(model, validation_loader)
        rmse = validation["cnn"]["rmse_pawns"]
        if not np.isfinite(rmse):
            raise RuntimeError("Non-finite validation error; check labels and learning rate")
        if rmse < best_rmse:
            best_rmse = rmse
            best_state = deepcopy(model.state_dict())
            if checkpoint_path is not None:
                torch.save({"model_state_dict": best_state, "epoch": epoch + 1,
                            "validation": validation}, checkpoint_path)
        history.append({"epoch": epoch + 1,
                        "train_mse": total_loss / len(train_loader.dataset),
                        "validation": validation})
        print(json.dumps(history[-1]))
    # Select the checkpoint using validation only; test games are not used here.
    model.load_state_dict(best_state)
    return history

In [ ]:
history = train_model(model, optimizer, criterion,
                      loaders["train"], loaders["validation"], epochs=EPOCHS,
                      checkpoint_path=CHECKPOINT_PATH)

In [ ]:
# Run once after selecting the model; do not tune hyperparameters on test error.
test_metrics = evaluate(model, loaders["test"])
print("Held-out test metrics (pawns):")
print(json.dumps(test_metrics, indent=2))